# EASY-EP expert pruning on Kaggle (GPU T4 x2)

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Inputs (Add data):**
* a dataset with `calibration.jsonl` (`{"human": ..., "response": ...}` per line) and, optionally,
  `c4_support_eval.jsonl`. The profiles expect it at `/kaggle/input/easyep-data/`; change
  `DATA_DIR` below if your dataset has another name.
* *(OLMoE, optional but recommended)* a dataset with the float16 copy of OLMoE produced by
  section 3, so the 28 GB float32 download is not repeated every session.

Disk: `/kaggle/working` (20 GB) is kept after the session; `/kaggle/temp` is not counted but is
wiped. The profiles put the Hugging Face download and the layer cache in `/kaggle/temp` and use
**delta checkpoints** (bias vectors + manifest, a few MB) instead of full model copies.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())
!df -h /kaggle/working /kaggle/temp 2>/dev/null; free -g | head -2

In [ ]:
REPO = "https://github.com/alfey504/EASYEP-modified.git"   # private repo: see the README "Kaggle" section
BRANCH = "kaggle_adapted"
!git clone -q -b {BRANCH} {REPO} /kaggle/working/easy_ep2
%cd /kaggle/working/easy_ep2
!mkdir -p /kaggle/temp
# Reuses Kaggle's preinstalled CUDA torch; upgrades transformers to 5.x if needed.
!pip install -q -e .
!python -c "import transformers, accelerate; print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)"
!cp -n .env.example .env

## 2. Choose the model

* `configs/kaggle/olmoe-1b-7b-t4x2.env` - OLMoE-1B-7B-0125, float16, split over both T4s.
* `configs/kaggle/switch-base-8-t4.env` - switch-base-8, float32, one T4.

`EXTRA` holds per-session overrides (`--set KEY=VALUE`), e.g. other dataset paths.

In [ ]:
PROFILE = "configs/kaggle/olmoe-1b-7b-t4x2.env"      # or "configs/kaggle/switch-base-8-t4.env"
DATA_DIR = "/kaggle/input/easyep-data"
FP16_MODEL_DIR = ""   # e.g. "/kaggle/input/olmoe-1b-7b-0125-fp16" once section 3 has been done

EXTRA = (f" --set CALIBRATION_DATASET_PATH={DATA_DIR}/calibration.jsonl"
         f" --set EVALUATION_DATASET_PATH={DATA_DIR}/c4_support_eval.jsonl")
if FP16_MODEL_DIR:
    EXTRA += f" --set MODEL_PATH={FP16_MODEL_DIR}"
P = f"--profile {PROFILE}{EXTRA} --set REPLACEMENT_METHOD=drop"   # or reroute
print(P)

### (Optional) build the C4 support evaluation set here
Only if `c4_support_eval.jsonl` is not in your dataset. Streams C4, no full download.

In [ ]:
# !mkdir -p /kaggle/working/data
# EXTRA += " --set EVALUATION_DATASET_PATH=/kaggle/working/data/c4_support_eval.jsonl"; P = f"--profile {PROFILE}{EXTRA}"
# !python scripts/build_c4_support_eval.py {P} --set C4_MAX_SCANNED=300000

## 3. (OLMoE, once) save a float16 copy of the model
Downloads the 28 GB float32 checkpoint to `/kaggle/temp`, writes a 14 GB float16 copy to
`/kaggle/working`. Then **Save Version**, create a dataset from this notebook's output, attach it,
and set `FP16_MODEL_DIR` above. Skip for switch-base-8.

In [ ]:
# !python scripts/convert_model.py {P} --output /kaggle/working/olmoe-1b-7b-0125-fp16
# !rm -rf /kaggle/temp/hf      # free the float32 download

## 4. Pipeline
Each step saves its results under `/kaggle/working/outputs`, so a later session can continue from
any step. `inspect_model.py` stops with a clear message if float16 overflows.

In [ ]:
!python scripts/inspect_model.py {P}

In [ ]:
!python scripts/collect_stats.py {P}

In [ ]:
!python scripts/score_experts.py {P}

In [ ]:
# REPLACEMENT_METHOD: bias (default) | drop | reroute  -> add e.g. --set REPLACEMENT_METHOD=drop
!python scripts/replace_experts.py {P}

In [ ]:
!python scripts/evaluate.py --stage before_training {P}

In [ ]:
# bias method only; set BIAS_LEARNING_RATE relative to the bias norms in bias_initialization.json
!python scripts/train_biases.py {P}

In [ ]:
!python scripts/evaluate.py --stage after_training {P}

In [ ]:
# --format full writes a self-contained checkpoint (~14 GB for OLMoE); delta (default here) is a few MB
!python scripts/export_model.py {P}

## 5. Results

In [ ]:
import glob
from IPython.display import Markdown, display
for path in glob.glob("/kaggle/working/outputs/experiments/*/evaluation/comparison.md"):
    display(Markdown(f"**{path}**\n\n" + open(path).read()))

In [ ]:
# Free temporary disk (re-created on demand)
!rm -rf /kaggle/temp/cache
!du -sh /kaggle/working/outputs